In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import random

from sklearn.metrics import classification_report, confusion_matrix

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print("torch:", torch.__version__)

device: cuda
torch: 2.12.1+cu130


In [2]:
H, W = 4, 4
N = H * W

def node_id(r, c):
    return r * W + c

def coord(node):
    return divmod(node, W)

def build_mesh_adj(H=4, W=4):
    N = H * W
    A = torch.zeros((N, N), dtype=torch.float32)

    for r in range(H):
        for c in range(W):
            u = node_id(r, c)

            # connect east-west
            if c + 1 < W:
                v = node_id(r, c + 1)
                A[u, v] = 1
                A[v, u] = 1

            # connect north-south
            if r + 1 < H:
                v = node_id(r + 1, c)
                A[u, v] = 1
                A[v, u] = 1

    return A

def normalize_adj(A):
    # Add self-loops
    A_hat = A + torch.eye(A.shape[0])

    # D^(-1/2) A D^(-1/2)
    deg = A_hat.sum(dim=1)
    D_inv_sqrt = torch.diag(torch.pow(deg, -0.5))

    return D_inv_sqrt @ A_hat @ D_inv_sqrt

A = build_mesh_adj(H, W)
A_norm = normalize_adj(A).to(device)

print("Adjacency shape:", A.shape)
print(A)

Adjacency shape: torch.Size([16, 16])
tensor([[0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 1., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0.],


In [3]:
def manhattan_distance(a, b):
    ar, ac = coord(a)
    br, bc = coord(b)
    return abs(ar - br) + abs(ac - bc)

def make_labels(attacked_router):
    """
    label 0 = normal
    label 1 = second-hop affected
    label 2 = first-hop affected or attacked router
    """
    y = torch.zeros(N, dtype=torch.long)

    for node in range(N):
        d = manhattan_distance(node, attacked_router)

        if d <= 1:
            y[node] = 2
        elif d == 2:
            y[node] = 1
        else:
            y[node] = 0

    return y

attacked_router = 5
y = make_labels(attacked_router)

print("attacked router:", attacked_router)
print("labels:", y.reshape(H, W))

attacked router: 5
labels: tensor([[1, 2, 1, 0],
        [2, 2, 2, 1],
        [1, 2, 1, 0],
        [0, 1, 0, 0]])


In [4]:
NUM_FEATURES = 5

def make_features(y):
    """
    Fake traffic features per router:
    [in_flits, out_flits, avg_delay, max_delay, queue_level]
    """

    x = torch.zeros((N, NUM_FEATURES), dtype=torch.float32)

    for node in range(N):
        label = y[node].item()

        # normal baseline traffic
        base = np.random.normal(loc=0.2, scale=0.05, size=NUM_FEATURES)

        if label == 2:
            # attacked / first-hop: high traffic and high delay
            effect = np.array([0.7, 0.7, 0.6, 0.8, 0.7])
        elif label == 1:
            # second-hop: medium disturbance
            effect = np.array([0.35, 0.35, 0.3, 0.4, 0.35])
        else:
            # unaffected
            effect = np.array([0, 0, 0, 0, 0])

        noise = np.random.normal(loc=0.0, scale=0.03, size=NUM_FEATURES)
        features = base + effect + noise
        features = np.clip(features, 0.0, 1.0)

        x[node] = torch.tensor(features, dtype=torch.float32)

    return x

x = make_features(y)

print("x shape:", x.shape)
print(x[:5])

x shape: torch.Size([16, 5])
tensor([[0.5678, 0.5905, 0.5554, 0.6621, 0.5546],
        [0.8600, 0.8463, 0.8215, 0.8771, 0.7714],
        [0.6266, 0.5042, 0.5146, 0.5107, 0.5140],
        [0.1333, 0.2989, 0.1405, 0.1073, 0.2470],
        [0.9153, 0.8947, 0.8259, 0.9953, 0.7732]])


In [5]:
def make_dataset(num_samples=500):
    X = []
    Y = []

    for _ in range(num_samples):
        attacked_router = random.randint(0, N - 1)

        y = make_labels(attacked_router)
        x = make_features(y)

        X.append(x)
        Y.append(y)

    X = torch.stack(X)  # [num_samples, num_nodes, num_features]
    Y = torch.stack(Y)  # [num_samples, num_nodes]

    return X, Y

X, Y = make_dataset(800)

print("X:", X.shape)
print("Y:", Y.shape)

X: torch.Size([800, 16, 5])
Y: torch.Size([800, 16])


In [6]:
num_train = int(0.8 * len(X))

X_train = X[:num_train].to(device)
Y_train = Y[:num_train].to(device)

X_test = X[num_train:].to(device)
Y_test = Y[num_train:].to(device)

print("train:", X_train.shape, Y_train.shape)
print("test:", X_test.shape, Y_test.shape)

train: torch.Size([640, 16, 5]) torch.Size([640, 16])
test: torch.Size([160, 16, 5]) torch.Size([160, 16])


In [7]:
class SimpleGCN(nn.Module):
    def __init__(self, in_features, hidden_dim, num_classes):
        super().__init__()

        self.lin1 = nn.Linear(in_features, hidden_dim)
        self.lin2 = nn.Linear(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, A_norm):
        """
        x shape: [batch, num_nodes, num_features]
        A_norm shape: [num_nodes, num_nodes]
        """

        # GNN layer 1: aggregate 1-hop neighbors
        h = torch.matmul(A_norm, x)
        h = self.lin1(h)
        h = F.relu(h)

        # GNN layer 2: aggregate 2-hop information
        h = torch.matmul(A_norm, h)
        h = self.lin2(h)
        h = F.relu(h)

        # node classifier
        out = self.classifier(h)

        return out

In [8]:
model = SimpleGCN(
    in_features=NUM_FEATURES,
    hidden_dim=32,
    num_classes=3
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

for epoch in range(1, 201):
    model.train()

    optimizer.zero_grad()

    logits = model(X_train, A_norm)  # [batch, nodes, classes]

    loss = criterion(
        logits.reshape(-1, 3),
        Y_train.reshape(-1)
    )

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        model.eval()
        with torch.no_grad():
            test_logits = model(X_test, A_norm)
            preds = test_logits.argmax(dim=-1)

            acc = (preds == Y_test).float().mean().item()

        print(f"epoch {epoch:03d} | loss {loss.item():.4f} | test acc {acc:.4f}")

epoch 020 | loss 0.5887 | test acc 0.7367
epoch 040 | loss 0.2509 | test acc 0.9195
epoch 060 | loss 0.2148 | test acc 0.9246
epoch 080 | loss 0.2118 | test acc 0.9270
epoch 100 | loss 0.2080 | test acc 0.9289
epoch 120 | loss 0.2033 | test acc 0.9305
epoch 140 | loss 0.1987 | test acc 0.9297
epoch 160 | loss 0.1931 | test acc 0.9328
epoch 180 | loss 0.1883 | test acc 0.9371
epoch 200 | loss 0.1821 | test acc 0.9367


In [9]:
model.eval()

with torch.no_grad():
    logits = model(X_test, A_norm)
    preds = logits.argmax(dim=-1)

y_true = Y_test.cpu().numpy().reshape(-1)
y_pred = preds.cpu().numpy().reshape(-1)

print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["normal", "second-hop", "first-hop/attacked"]
))

Confusion matrix:
[[1168   81    0]
 [  41  594   39]
 [   0    1  636]]

Classification report:
                    precision    recall  f1-score   support

            normal       0.97      0.94      0.95      1249
        second-hop       0.88      0.88      0.88       674
first-hop/attacked       0.94      1.00      0.97       637

          accuracy                           0.94      2560
         macro avg       0.93      0.94      0.93      2560
      weighted avg       0.94      0.94      0.94      2560



In [10]:
sample_id = 0

true_grid = Y_test[sample_id].cpu().reshape(H, W)
pred_grid = preds[sample_id].cpu().reshape(H, W)

print("True labels:")
print(true_grid)

print("\nPredicted labels:")
print(pred_grid)

True labels:
tensor([[2, 1, 0, 0],
        [2, 2, 1, 0],
        [2, 1, 0, 0],
        [1, 0, 0, 0]])

Predicted labels:
tensor([[2, 1, 0, 0],
        [2, 2, 1, 0],
        [2, 1, 0, 0],
        [1, 0, 0, 0]])
